# Transfer Format Analysis

Visualize results from the `transfer_format` benchmark suite.

This suite tests how different image transfer formats (none, pil, png, jpeg)
affect steganographic accuracy when the D calculation is matched to the format.

**Run the benchmark first:**
```bash
python benchmark_ldstega.py --suite transfer_format --seeds 42,123,777
```

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import json
from pathlib import Path

sns.set_theme(style="whitegrid", font_scale=1.2)
plt.rcParams["figure.dpi"] = 120

CSV_PATH = Path("../results/full/benchmark_results.csv")
assert CSV_PATH.exists(), f"Results not found at {CSV_PATH}. Run benchmark_ldstega.py first."

df_all = pd.read_csv(CSV_PATH)
#df_all = df_all[df_all.seed == 42]
# Filter to transfer_format suite
df = df_all[df_all["suite"] == "transfer_format"].copy()
assert len(df) > 0, "No transfer_format suite results found. Run: python benchmark_ldstega.py --suite transfer_format"

# Parse transfer_format from transform_params or column
if "transfer_format" in df.columns:
    df["format"] = df["transfer_format"]
else:
    df["transform_params_parsed"] = df["transform_params"].apply(json.loads)
    df["format"] = df["transform_params_parsed"].apply(lambda x: x.get("transfer_format", "pil"))

# Clean format labels for display
df["format_label"] = df["format"].str.replace(":", " q", regex=False)

# Ensure message_chars column exists
if "message_chars" not in df.columns:
    df["message_chars"] = df["message_length"] // 8

print(f"Loaded {len(df)} rows from transfer_format suite")
print(f"Formats: {sorted(df['format'].unique())}")
print(f"Gammas: {sorted(df['gamma'].unique())}")
print(f"Message lengths: {sorted(df['message_length'].unique())}")
print(f"Seeds: {sorted(df['seed'].unique())}")

## 1. bit_accuracy Heatmap: Format x Gamma

Averaged over message lengths and seeds. Shows which formats are most robust at each gamma.

In [ ]:
agg_fg = (
    df.groupby(["format_label", "gamma"])["bit_accuracy"]
    .agg(["mean", "std"]).reset_index()
)
pivot_fg = agg_fg.pivot(index="format_label", columns="gamma", values="mean")
pivot_std = agg_fg.pivot(index="format_label", columns="gamma", values="std")

row_order = [l for l in ["none", "pil", "png", "jpeg q95", "jpeg q75", "jpeg q50"] if l in pivot_fg.index]
pivot_fg = pivot_fg.reindex(row_order)
pivot_std = pivot_std.reindex(row_order)

annot = pivot_fg.map(lambda v: f"{v:.3f}") + "\n±" + pivot_std.map(lambda v: f"{v:.3f}")

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(
    pivot_fg, annot=annot, fmt="", cmap="RdYlGn",
    vmin=0, vmax=max(0.5, pivot_fg.max().max()),
    linewidths=0.5, ax=ax,
    cbar_kws={"label": "bit_accuracy (mean)"},
    annot_kws={"size": 10},
)
ax.set_title("bit_accuracy by Transfer Format and Gamma\n(mean ± std over message lengths & seeds)")
ax.set_ylabel("Transfer Format")
ax.set_xlabel("Gamma (γ)")
plt.tight_layout()
plt.savefig("../results/tf_format_x_gamma_heatmap.png", bbox_inches="tight")
plt.show()

## 2. bit_accuracy Heatmap: Format x Message Length

Averaged over gammas and seeds. Shows how capacity scales per format.

In [ ]:
tmp = df[df.gamma == 0.3].copy()
tmp.seed.value_counts()
pivot_fm = (
    tmp.groupby(["format_label", "message_length"])["bit_accuracy"].mean().reset_index()
    .pivot(index="format_label", columns="message_length", values="bit_accuracy")
)
row_order = [l for l in ["none", "pil", "png", "jpeg q95", "jpeg q75", "jpeg q50"] if l in pivot_fm.index]
pivot_fm = pivot_fm.reindex(row_order)

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(
    pivot_fm, annot=True, fmt=".3f", cmap="RdYlGn",
    vmin=0, vmax=max(0.5, pivot_fm.max().max()),
    linewidths=0.5, ax=ax,
    cbar_kws={"label": "bit_accuracy"},
)
ax.set_title("bit_accuracy by Transfer Format and Message Length (averaged over gammas)")
ax.set_ylabel("Transfer Format")
ax.set_xlabel("Message Length (bits)")
plt.tight_layout()
plt.savefig("../results/tf_format_x_msglen_heatmap.png", bbox_inches="tight")
plt.show()

## 3. bit_accuracy vs Message Length per Format

Shows the capacity degradation curve for each transfer format.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

format_order = ["none", "pil", "png", "jpeg:95", "jpeg:75", "jpeg:50"]
colors = sns.color_palette("tab10", n_colors=len(format_order))
markers = ["o", "s", "D", "^", "v", "<"]
tmp = df[df.gamma == 0.3].copy()
for i, fmt in enumerate(format_order):
    sub = tmp[tmp["format"] == fmt]
    if len(sub) == 0:
        continue
    agg = sub.groupby("message_length")["bit_accuracy"].agg(["mean", "std"]).reset_index()
    ax.errorbar(
        agg["message_length"], agg["mean"], yerr=agg["std"],
        marker=markers[i], capsize=4, linewidth=2, color=colors[i],
        label=fmt.replace(":", " q"),
    )

ax.set_xlabel("Message Length (bits)")
ax.set_ylabel("bit_accuracy")
ax.set_title("bit_accuracy vs Message Length by Transfer Format")
ax.axhline(y=0.5, color="red", linestyle="--", alpha=0.3, label="Random guess")
ax.legend(fontsize=9)
ax.set_xscale("log", base=2)
ax.xaxis.set_major_formatter(mticker.ScalarFormatter())
plt.tight_layout()
plt.savefig("../results/tf_bit_accuracy_vs_msglen.png", bbox_inches="tight")
plt.show()

## 4. bit_accuracy vs Gamma per Format

Shows sensitivity to the truncation parameter for each format.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for i, fmt in enumerate(format_order):
    sub = df[df["format"] == fmt]
    if len(sub) == 0:
        continue
    agg = sub.groupby("gamma")["bit_accuracy"].agg(["mean", "std"]).reset_index()
    ax.errorbar(
        agg["gamma"], agg["mean"], yerr=agg["std"],
        marker=markers[i], capsize=4, linewidth=2, color=colors[i],
        label=fmt.replace(":", " q"),
    )

ax.set_xlabel("Gamma (\u03b3)")
ax.set_ylabel("bit_accuracy")
ax.set_title("bit_accuracy vs Gamma by Transfer Format")
ax.axhline(y=0.5, color="red", linestyle="--", alpha=0.3, label="Random guess")
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig("../results/tf_bit_accuracy_vs_gamma.png", bbox_inches="tight")
plt.show()

## 5. Direct Format Comparison (gamma=0.3, msg_len=512)

Bar chart at a standard operating point for direct comparison.

In [ ]:
# Pick the closest available gamma and msg_len to 0.3 / 512
target_gamma = 0.3
target_msglen = 512

sub = df[(df["gamma"] == target_gamma) & (df["message_length"] == target_msglen)]

if len(sub) > 0:
    fig, ax = plt.subplots(figsize=(10, 5))

    order = [l.replace(":", " q") for l in format_order if l in sub["format"].values]
    palette = sns.color_palette("tab10", n_colors=len(order))

    sns.barplot(
        data=sub, x="format_label", y="bit_accuracy", order=order,
        palette=palette, errorbar="sd", capsize=0.1, ax=ax,
    )

    ax.set_ylabel("bit_accuracy")
    ax.set_xlabel("Transfer Format")
    ax.set_title(f"bit_accuracy by Transfer Format (\u03b3={target_gamma}, {target_msglen} bits)")

    # Annotate bars
    for i, name in enumerate(order):
        mean_bit_accuracy = sub[sub["format_label"] == name]["bit_accuracy"].mean()
        ax.text(i, mean_bit_accuracy + 0.005, f"{mean_bit_accuracy:.4f}", ha="center", fontsize=9)

    plt.tight_layout()
    plt.savefig("../results/tf_bar_comparison.png", bbox_inches="tight")
    plt.show()
else:
    print(f"No data for gamma={target_gamma}, msg_len={target_msglen}")

## 6. PSNR / SSIM vs bit_accuracy by Format

Shows image quality vs accuracy tradeoff, colored by transfer format.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, fmt in enumerate(format_order):
    sub = df[df["format"] == fmt]
    if len(sub) == 0:
        continue
    label = fmt.replace(":", " q")
    axes[0].scatter(
        sub["psnr_stego"], sub["bit_accuracy"],
        alpha=0.5, s=20, color=colors[i], label=label,
    )
    axes[1].scatter(
        sub["ssim_stego"], sub["bit_accuracy"],
        alpha=0.5, s=20, color=colors[i], label=label,
    )

axes[0].set_xlabel("PSNR (stego vs original) [dB]")
axes[0].set_ylabel("bit_accuracy")
axes[0].set_title("PSNR vs bit_accuracy")
axes[0].axhline(y=0.5, color="red", linestyle="--", alpha=0.3)
axes[0].legend(fontsize=8)

axes[1].set_xlabel("SSIM (stego vs original)")
axes[1].set_ylabel("bit_accuracy")
axes[1].set_title("SSIM vs bit_accuracy")
axes[1].axhline(y=0.5, color="red", linestyle="--", alpha=0.3)
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig("../results/tf_psnr_ssim_vs_bit_accuracy.png", bbox_inches="tight")
plt.show()

## 7. Summary Table

Best operating point per format and maximum capacity at >=95% bit accuracy.

In [ ]:
# Mean bit_accuracy per format
summary = (
    df.groupby("format_label")
    .agg(
        bit_accuracy_mean=("bit_accuracy", "mean"),
        bit_accuracy_std=("bit_accuracy", "std"),
        bit_acc_mean=("bit_accuracy", "mean"),
        psnr_stego_mean=("psnr_stego", "mean"),
        ssim_stego_mean=("ssim_stego", "mean"),
    )
    .round(4)
    .sort_values("bit_accuracy_mean")
)
print("Overall bit_accuracy by Format:")
print("=" * 72)
display(summary)

# Max capacity at >= 95% bit accuracy per format
print("\nMax message length at >= 95% bit accuracy (mean across seeds):")
print("-" * 60)
for fmt in format_order:
    label = fmt.replace(":", " q")
    sub = df[df["format"] == fmt]
    if len(sub) == 0:
        continue
    # Mean bit accuracy per (gamma, msg_len)
    agg = sub.groupby(["gamma", "message_length"])["bit_accuracy"].mean().reset_index()
    good = agg[agg["bit_accuracy"] >= 0.95]
    if len(good) > 0:
        best = good.loc[good["message_length"].idxmax()]
        print(f"  {label:12s}  {int(best['message_length']):5d} bits  "
              f"(gamma={best['gamma']}, acc={best['bit_accuracy']:.3f})")
    else:
        print(f"  {label:12s}  No config reaches 95% accuracy")

# Best operating point per format (lowest bit_accuracy)
print("\nBest operating point per format (lowest mean bit_accuracy):")
print("-" * 60)
for fmt in format_order:
    label = fmt.replace(":", " q")
    sub = df[df["format"] == fmt]
    if len(sub) == 0:
        continue
    agg = sub.groupby(["gamma", "message_length"])["bit_accuracy"].mean().reset_index()
    best_idx = agg["bit_accuracy"].idxmin()
    best = agg.loc[best_idx]
    print(f"  {label:12s}  gamma={best['gamma']}, len={int(best['message_length'])}, "
          f"bit_accuracy={best['bit_accuracy']:.4f}")